# `flattening_tests` — Silver Customers Pipeline Tests

Validates that:
1. The `silver_customers` notebook executes without errors.
2. The `dbo.silver_customers` table exists and contains records.
3. No column contains JSON-like values (strings with `{…}` / `[…]`).
4. The schema contains no complex types (`StructType`, `MapType`, `ArrayType`).


In [ ]:
# ============================================================
# 1. IMPORTS & CONFIGURATION
# ============================================================

import json
import re
import traceback

import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, MapType, StringType, StructType

# Table under test
SILVER_TABLE = "dbo.silver_customers"

# Rows to sample when probing for JSON values
SAMPLE_SIZE = 200

# ── Test registry ────────────────────────────────────────────
_results: list[dict] = []

def _record(table: str, column, test: str, passed: bool, detail: str = "") -> None:
    status = "✅ PASS" if passed else "❌ FAIL"
    _results.append({"table": table, "column": column or "—", "test": test, "status": status, "detail": detail})
    icon = "✅" if passed else "❌"
    col_label = f" [{column}]" if column else ""
    print(f"  {icon} {test}{col_label}{': ' + detail if detail else ''}")


In [ ]:
# ============================================================
# 3. VALIDATE TABLE EXISTS & HAS ROWS
# ============================================================

print("── Validating table existence & row count ──────────────")

_table_exists = spark.catalog.tableExists(SILVER_TABLE)
_record(SILVER_TABLE, None, "Table exists", _table_exists)

if _table_exists:
    df_silver = spark.read.table(SILVER_TABLE)
    _row_count = df_silver.count()
    _record(SILVER_TABLE, None, "Table is not empty", _row_count > 0, detail=f"{_row_count:,} rows")
    print(f"  Schema:\n{df_silver._jdf.schema().treeString()}")
else:
    df_silver = None
    print("  Skipping further checks — table does not exist.")


In [ ]:
# ============================================================
# 4. SCHEMA-LEVEL: NO COMPLEX TYPES (Struct/Map/Array)
# ============================================================

print("── Checking schema for complex (non-flat) types ────────")

COMPLEX_TYPES = (StructType, ArrayType, MapType)

def _check_schema_flat(schema, prefix=""):
    """Recursively walk schema; yield (full_field_name, type_name) for any complex field."""
    for field in schema.fields:
        full_name = f"{prefix}.{field.name}" if prefix else field.name
        dt = field.dataType
        if isinstance(dt, COMPLEX_TYPES):
            yield full_name, type(dt).__name__
        if isinstance(dt, StructType):
            yield from _check_schema_flat(dt, prefix=full_name)

if df_silver:
    complex_fields = list(_check_schema_flat(df_silver.schema))
    if complex_fields:
        for fname, ftype in complex_fields:
            _record(SILVER_TABLE, fname, "Column is not a complex type", False, detail=ftype)
    else:
        _record(SILVER_TABLE, None, "No complex schema types (Struct/Map/Array)", True)


In [ ]:
# ============================================================
# 5. VALUE-LEVEL: NO JSON-LIKE STRINGS IN STRING COLUMNS
# ============================================================

print("── Sampling string columns for JSON-like values ────────")

_JSON_RE = re.compile(r"^\s*[\{\[]")   # starts with { or [

def _looks_like_json(value):
    """Return True if value appears to be a JSON object or array."""
    if not isinstance(value, str):
        return False
    if _JSON_RE.match(value):
        # Confirm it actually parses as JSON
        try:
            json.loads(value)
            return True
        except (json.JSONDecodeError, ValueError):
            pass
    return False


if df_silver:
    string_cols = [f.name for f in df_silver.schema.fields if isinstance(f.dataType, StringType)]
    print(f"  String columns to check: {string_cols}")

    for col_name in string_cols:
        # Backtick-escape column names with special chars (dots, spaces, etc.)
        col_ref = F.col(f"`{col_name}`")
        
        try:
            sample_vals = [
                row[0]
                for row in (
                    df_silver.select(col_ref)
                    .filter(col_ref.isNotNull())
                    .limit(SAMPLE_SIZE)
                    .collect()
                )
            ]
            offenders = [v for v in sample_vals if _looks_like_json(v)]
            passed    = len(offenders) == 0
            detail    = f"sample: {offenders[0][:80]}…" if offenders else ""
            _record(SILVER_TABLE, col_name, "Column contains no JSON-like strings", passed, detail=detail)
        except Exception as e:
            _record(SILVER_TABLE, col_name, "Column contains no JSON-like strings", False, 
                    detail=f"Error sampling column: {str(e)[:80]}")


In [ ]:
# ============================================================
# 6. ASSERT: Upsert key column integrity
# ============================================================

print("── Validating upsert key integrity ─────────────────────")

if df_silver:
    _upsert_key = "CustomerId"
    if _upsert_key in df_silver.columns:
        # Backtick-escape for safety, even if not strictly needed here
        key_col = F.col(f"`{_upsert_key}`")
        
        null_count = df_silver.filter(key_col.isNull()).count()
        _record(SILVER_TABLE, _upsert_key, "Upsert key has no NULL values", null_count == 0,
                detail=f"{null_count} NULLs found" if null_count else "")

        dup_count = (df_silver.groupBy(key_col).count()
                     .filter(F.col("count") > 1).count())
        _record(SILVER_TABLE, _upsert_key, "Upsert key has no duplicates", dup_count == 0,
                detail=f"{dup_count} duplicate key(s)" if dup_count else "")
    else:
        _record(SILVER_TABLE, _upsert_key, f"Column '{_upsert_key}' exists", False)

# Hard-fail on any FAIL result so the notebook cell errors on problems
failed = [r for r in _results if "FAIL" in r["status"]]
if failed:
    print(f"\n⚠️  {len(failed)} test(s) FAILED — see summary below.")
else:
    print("\n✅  All assertions passed.")


In [ ]:
# ============================================================
# 7. TEST SUMMARY REPORT
# ============================================================

import pandas as pd

df_summary = pd.DataFrame(_results, columns=["table", "column", "test", "status", "detail"])

def _highlight_status(val):
    if "FAIL" in str(val):
        return "background-color: #ffcccc; color: #8b0000; font-weight: bold"
    if "PASS" in str(val):
        return "background-color: #ccffcc; color: #1a6b1a; font-weight: bold"
    return ""

total  = len(df_summary)
passed = (df_summary["status"].str.contains("PASS")).sum()
failed = total - passed

print(f"\n{'='*60}")
print(f"  TEST SUMMARY   {passed}/{total} passed   {failed} failed")
print(f"{'='*60}\n")

styled = (
    df_summary.style
    .applymap(_highlight_status, subset=["status"])
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
    .hide(axis="index")
)

display(styled)

# Hard-fail the cell if any test failed — surfaces in pipeline orchestration
assert failed == 0, f"{failed} test(s) FAILED. See table above."
